In [4]:
# INSTALLATION: pip install transformers==4.48.0

import torch
from transformers import pipeline

# Load the model
model_id = "Qualcomm-AI-Research/PhoASR-whisper-small"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Create pipeline for automatic speech recognition with word-level timestamps
pipe = pipeline("automatic-speech-recognition", model=model_id, chunk_length_s=30, device=device, return_timestamps="word", generate_kwargs={"language": "vi"})


Loading weights: 100%|██████████| 479/479 [00:00<00:00, 8143.82it/s]
[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


In [5]:
# Transcribe a single audio file
audio = r"G:\Uni\Sem2\NLP\term_paper\data\ViSQA\ViSQA\speech_data\train\contexts\ctx_0d77e608.mp3"

In [3]:
result = pipe(audio)

# Print full transcription
print(f"Full Text: {result['text']}")

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


KeyboardInterrupt: 

In [6]:
import soundfile as sf
import torch
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

model_id = "Qualcomm-AI-Research/PhoASR-whisper-small"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id).to(device)


audio, sampling_rate = sf.read(audio)

# Convert stereo -> mono
if audio.ndim > 1:
    audio = audio.mean(axis=1)

# Whisper expects 16 kHz
if sampling_rate != 16000:
    import librosa
    audio = librosa.resample(
        audio.astype("float32"),
        orig_sr=sampling_rate,
        target_sr=16000,
    )
    sampling_rate = 16000

inputs = processor(
    audio,
    sampling_rate=sampling_rate,
    return_tensors="pt",
)

inputs = {k: v.to(device) for k, v in inputs.items()}

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 8336.57it/s]


In [ ]:
generated = model.generate(
    **inputs,
    language="vi",
    task="transcribe",

    do_sample=False,
    num_beams=1,
    num_return_sequences=1,

    return_dict_in_generate=True,
    output_scores=True,
)

hypotheses = processor.batch_decode(
    generated,
    skip_special_tokens=True,
)

for i, text in enumerate(hypotheses, 1):
    print(f"{i}: {text}")

[transformers] The following generation flags are not valid and may be ignored: ['output_scores']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [5]:
from datasets import load_dataset

ds = load_dataset("taidng/UIT-ViQuAD2.0")

g:\Uni\Sem2\NLP\term_paper\nemo_312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in G:\.cache\huggingface\hub\datasets--taidng--UIT-ViQuAD2.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 7301/7301 [00:00<00:00, 313278.02 examples/s]
